# Mixed mechanics locking qualification

Candidate verification, not a pre-existing success report. Runs pinned code in an isolated environment, repeats the original 80.2% failure alongside MINI and retains negative evidence. The original 15% screen is unchanged. No patient data. Read `docs/MIXED_MINI_DIAGNOSIS.md` and `docs/MIXED_MINI_ACCEPTANCE.md`.


In [ ]:
import sys, os, time, pathlib, subprocess, json, hashlib, zipfile
ROOT = pathlib.Path('/content/CardiMech_mixed_'+time.strftime('%Y%m%d_%H%M%S'))
ROOT.mkdir()
CODE_COMMIT = 'beaf842e5f6518e74b69b03c40ece7f9ca8e0f93'
stages = {}
def run(args, name, cwd=None, timeout=7200):
    print('START', name, flush=True)
    try:
        p = subprocess.run(args, cwd=cwd, env=ENV if 'ENV' in globals() else None,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, timeout=timeout)
    except subprocess.TimeoutExpired as exc:
        raw = exc.stdout or ''
        (ROOT/name).write_text(raw.decode(errors='replace') if isinstance(raw,bytes) else raw)
        stages[name] = {'passed':False,'timeout':timeout}
        raise
    (ROOT/name).write_text(p.stdout)
    stages[name] = {'passed':p.returncode==0,'returncode':p.returncode}
    print(name,p.returncode,p.stdout[-4000:],flush=True)
    p.check_returncode()
source_paths = ["src/cardimech/fem_mixed.py","src/cardimech/fem_backend.py","tests/test_fem_mixed.py","tests/test_fem_material_regime.py","scripts/qualify_material_regimes.py","scripts/diagnose_locking.py","scripts/verify_mixed_locking.py","docs/MIXED_MINI_ACCEPTANCE.md","docs/MIXED_MINI_DIAGNOSIS.md"]
failure = None
try:
    run([sys.executable,'-m','pip','install','virtualenv==21.14.12'],'bootstrap.log')
    run([sys.executable,'-m','virtualenv',str(ROOT/'venv')],'venv.log')
    PY = str(ROOT/'venv/bin/python')
    ENV = os.environ.copy()
    ENV['PATH'] = str(ROOT/'venv/bin')+os.pathsep+ENV['PATH']
    ENV['OPENBLAS_NUM_THREADS'] = '1'
    REPO = ROOT/'CardiMech'
    run(['git','clone','https://github.com/Virelion-Biotech/Virelion-CardiMech.git',str(REPO)],'clone.log')
    run(['git','checkout',CODE_COMMIT],'checkout.log',REPO)
    run([PY,'-m','pip','install','numpy==2.1.3','scipy==1.16.3','pydantic==2.14.1',
         'pytest==8.4.2','ruff==0.17.1','-e',str(REPO)+'[dev,validation]',
         'git+https://github.com/Virelion-Biotech/Virelion-CardiInfer.git@9aabf87bdbee4c70b98229baa3bbea3fb896b490'],'install.log')
    run([PY,'-m','ruff','check','src','tests','scripts'],'lint.log',REPO)
    run([PY,'-m','ruff','format','--check','src','tests','scripts'],'format.log',REPO)
    run([PY,'-m','pytest','-q','--cov=cardimech','--cov-fail-under=80',
         '--junitxml='+str(ROOT/'suite.xml')],'suite.log',REPO)
    run([PY,'scripts/diagnose_locking.py','--output',str(ROOT/'linear-diagnosis')],'linear.log',REPO)
    run([PY,'scripts/verify_mixed_locking.py','--output',str(ROOT/'campaign')],'campaign.log',REPO)
    run([PY,'-m','pip','freeze'],'environment.txt',REPO)
except Exception as exc:
    failure = {'type':type(exc).__name__,'message':str(exc)}
finally:
    qualification_path = ROOT/'campaign/qualification.json'
    qualification = json.loads(qualification_path.read_text()) if qualification_path.exists() else None
    identity = {p:hashlib.sha256((REPO/p).read_bytes()).hexdigest() for p in source_paths
                if 'REPO' in globals() and (REPO/p).exists()}
    manifest = {'code_commit':CODE_COMMIT,'stages':stages,'failure':failure,
                'source_sha256':identity,'qualification':qualification,
                'accuracy_qualified':bool(qualification and qualification['passed'] and not failure),
                'scope':'Synthetic numerical qualification only; no patient/whole-heart/clinical accuracy claim',
                'archive_contents':'All generated case meshes/fields, logs, JUnit, qualification and exact source; no runtime environment directories'}
    (ROOT/'MANIFEST.json').write_text(json.dumps(manifest,indent=2)+'\n')
    archive = ROOT/'mixed_locking_evidence.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in ROOT.rglob('*'):
            if p.is_file() and p != archive and not p.is_relative_to(ROOT/'venv') and not p.is_relative_to(ROOT/'CardiMech'):
                z.write(p,str(p.relative_to(ROOT)))
        for p in source_paths:
            if 'REPO' in globals() and (REPO/p).exists(): z.write(REPO/p,'source/'+p)
    raw = archive.read_bytes()
    receipt = {'bytes':len(raw),'sha256':hashlib.sha256(raw).hexdigest()}
    (ROOT/'receipt.json').write_text(json.dumps(receipt,indent=2)+'\n')
    print('EVIDENCE',json.dumps({'root':str(ROOT),'receipt':receipt,'accuracy_qualified':manifest['accuracy_qualified'],'failure':failure}),flush=True)
if failure:
    raise RuntimeError('Qualification did not pass; inspect retained logs and archive: '+str(failure))
assert manifest['accuracy_qualified']
print('MIXED_QUALIFICATION_PASSED',flush=True)
